In [1]:
from expyfun._utils import _TempDir
from expyfun import ExperimentController, analyze, building_doc
from expyfun.stimuli import (crm_prepare_corpus, crm_sentence, crm_info,
                             crm_response_menu, add_pad, CRMPreload)

import numpy as np
import pickle

In [2]:
import pathlib
from argparse import ArgumentParser
import yaml
import torch
import torchmetrics
import pytorch_lightning as pl
from tqdm import tqdm

/om4/group/mcdermott/user/imgriff/conda_envs_files/torch_11_cuda_11/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import sys 
sys.path.append('../')
from src.attn_tracking_lightning import AttentionalTrackingModule
from corpus.jsinV3AttnTrackingValidation import jsinV3_attn_tracking_validation
import src.audio_transforms as at


In [5]:
pl.seed_everything(1) 

Global seed set to 1


1

## Get CRM pieces & demo use

In [6]:

# crm_path = _TempDir()
fs = 20000

In [9]:
# Prepare CRM for DNN experiments

crm_path = "/om2/user/imgriff/datasets/CRM"

In [12]:
crm_info()

{'sex': ['male', 'female'],
 'talker_number': ['0', '1', '2', '3'],
 'callsign': ['charlie',
  'ringo',
  'laker',
  'hopper',
  'arrow',
  'tiger',
  'eagle',
  'baron'],
 'color': ['blue', 'red', 'white', 'green'],
 'number': ['1', '2', '3', '4', '5', '6', '7', '8']}

In [11]:

# print the valid callsigns
print('Valid callsigns are {0}'.format(crm_info()['callsign']))

# read a sentence in from the hard drive
x1 = 0.5 * crm_sentence(fs, 'm', '0', 'c', 'r', '5', path=crm_path)

# preload all the talkers and get a second sentence from memory
crm = CRMPreload(fs, path=crm_path)
x2 = crm.sentence('f', '0', 'ringo', 'green', '6')

x = add_pad([x1, x2], alignment='start')

Valid callsigns are ['charlie', 'ringo', 'laker', 'hopper', 'arrow', 'tiger', 'eagle', 'baron']


In [17]:
from IPython.display import Audio 

In [18]:
Audio(x, rate=fs)

In [11]:
crm_info()

{'sex': ['male', 'female'],
 'talker_number': ['0', '1', '2', '3'],
 'callsign': ['charlie',
  'ringo',
  'laker',
  'hopper',
  'arrow',
  'tiger',
  'eagle',
  'baron'],
 'color': ['blue', 'red', 'white', 'green'],
 'number': ['1', '2', '3', '4', '5', '6', '7', '8']}

## Load & Set up model

In [151]:
path = '../config/attentional_cue/attn_cue_lr_1e-4_bs_64_constrain_slope.yml'
config = yaml.load(open(path, 'r'), Loader=yaml.FullLoader)


audio_config = config['data']['audio']

audio_transforms = at.AudioCompose([
            at.AudioToTensor(),
            at.CombineWithRandomDBSNR(low_snr=10, high_snr=10), # set to 0 so foreground/background at same level 
            at.RMSNormalizeForegroundAndBackground(rms_level=0.1),

])
    
    
cochgram_transforms = at.AudioCompose([
            at.AudioToAudioRepresentation(**audio_config),
            at.UnsqueezeAudio(dim=0),
            at.UnsqueezeAudio(dim=0),


])

model = AttentionalTrackingModule(config)
# dataset = jsinV3_attn_tracking_validation(**config['data']['corpus'],
#                                           train=False,
#                                           transform=audio_transforms,
#                                           demo=True)

word_encodings = pickle.load( open( "/om4/group/mcdermott/user/jfeather/projects/model_metamers/figure_generation_notebooks/word_and_speaker_encodings_jsinv3.pckl", "rb" )) 
class_map = word_encodings['word_idx_to_word']
    
    

In [2]:
word_encodings = pickle.load( open( "/om4/group/mcdermott/user/jfeather/projects/model_metamers/figure_generation_notebooks/word_and_speaker_encodings_jsinv3.pckl", "rb" )) 
class_map = word_encodings['word_idx_to_word']
    

In [26]:
class_map.values()

dict_values(['__nullSignal__', 'ability', 'about', 'above', 'accepted', 'according', 'account', 'across', 'action', 'active', 'activities', 'activity', 'actually', 'added', 'addition', 'additional', 'advanced', 'africa', 'after', 'again', 'against', 'agency', 'agreed', 'agreement', 'allow', 'allowed', 'almost', 'alone', 'along', 'already', 'alternative', 'although', 'always', 'america', 'american', 'americans', 'among', 'amount', 'analysts', 'ancient', 'animals', 'announced', 'annual', 'another', 'appear', 'appearance', 'appeared', 'appears', 'appointed', 'approach', 'approximately', 'april', 'areas', 'around', 'article', 'asked', 'assets', 'associated', 'association', 'attack', 'attacks', 'attempt', 'attempted', 'attended', 'attention', 'august', 'authority', 'available', 'avoid', 'award', 'banks', 'based', 'basis', 'battle', 'became', 'because', 'become', 'becoming', 'before', 'began', 'beginning', 'behind', 'being', 'believe', 'believed', 'below', 'better', 'between', 'beyond', 'bil

Check how many sentences have target words in the model's vocabulary

In [219]:
num2word_map = ["one","two","three","four", "five", "six","seven","eight"]


valid_callsigns = [callsign for callsign in crm_info()['callsign'] if callsign in class_map.values()]
print(f"Valid callsigns: {valid_callsigns}")
valid_colors = [color for color in crm_info()['color'] if color in class_map.values()]
print(f"Valid colors: {valid_colors}")

valid_numbers = [num2word_map[int(number)-1] for number in crm_info()['number'] 
                 if num2word_map[int(number)-1] in class_map.values() or number in class_map.values()]
print(f"Valid numbers: {valid_numbers}")



Valid callsigns: []
Valid colors: ['white']
Valid numbers: ['three', 'seven', 'eight']


In [220]:
ckpt_path = '../attn_cue_models/attn_cue_jsin_pilot_no_pretrain_pos_slope_bs_64_lr_1e-4/checkpoints/epoch=1-step=145791.ckpt'
ckpt = torch.load(ckpt_path, map_location=torch.device('cpu'))

model.load_state_dict(ckpt['state_dict'])

<All keys matched successfully>

In [221]:
model = model.eval()


In [243]:


# read a sentence in from the hard drive

# preload all the talkers and get a second sentence from memory
crm = CRMPreload(fs, path=crm_path)
foreground = crm.sentence('f', '0', 'ringo', 'white', '3').astype('float32')
background = crm.sentence('m', '0', 'charlie', 'red', '5').astype('float32')

mixture_array = add_pad([foreground, background], alignment='start')

cue = crm.sentence('f', '0', 'laker', 'blue', '3').astype('float32')
bg_len = len(background)
fg_len =  len(foreground)
cue_len = len(cue)

max_len = 40000

if bg_len < max_len:
    background = np.pad(background, (0, (max_len - bg_len)), mode='constant', constant_values=0)

if fg_len < max_len:
    foreground = np.pad(foreground, (0, (max_len - fg_len)),  mode='constant', constant_values=0)
    
if cue_len < max_len:
    cue = np.pad(cue, (0, (max_len - cue_len)),  mode='constant', constant_values=0)

foreground = foreground[:max_len]
background = background[:max_len]
cue = cue[:max_len]


fg, _ = audio_transforms(foreground,None)
bg, _ = audio_transforms(background,None)
cue, _ = audio_transforms(cue, None)

mixture,_ = audio_transforms(foreground, background)

fg_tensor, _ = cochgram_transforms(fg,None)
bg_tensor, _ = cochgram_transforms(bg,None)
cue_tensor, _ = cochgram_transforms(cue, None)

mixture_tensor,_ = cochgram_transforms(mixture, None)


fg_tensor = fg_tensor[:,:,:, :16000]
bg_tensor = bg_tensor[:,:,:, :16000]
cue_tensor = cue_tensor[:,:,:, :16000]
mixture_tensor = mixture_tensor[:,:,:, :16000]

In [244]:
max_len - cue_len, cue_len

(3766, 36234)

In [245]:
cue_len

36234

In [246]:
background.shape, foreground.shape

((40000,), (40000,))

In [247]:
cue_t.shape

torch.Size([40000])

In [248]:
model_out = model(fg_tensor, mixture_tensor)
model_guesses = model_out.log_softmax(-1)

In [249]:
top_1 = model_guesses.argmax(-1).item()
top_5 = torch.topk(model_guesses, 10, dim=-1)
class_map[top_1]

'light'

In [250]:
Audio(cue, rate=20000)

In [251]:
Audio(mixture, rate=20000)

In [252]:
#guesses 
[class_map[ix.item()] for ix in top_5.indices[0]]

['light',
 'white',
 'likely',
 'might',
 'black',
 'right',
 'water',
 'avoid',
 'point',
 'would']